# 02 - Metric and Data Validation

## 1. Load processed data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"

CV_DIR = OUTPUTS / "cv_results"
CV_DIR.mkdir(parents=True, exist_ok=True)

daily_panel = pd.read_parquet(DATA_PROCESSED / "daily_panel.parquet")
sku_activity = pd.read_parquet(DATA_PROCESSED / "sku_activity.parquet")

daily_panel["Date"] = pd.to_datetime(daily_panel["Date"])

print("daily_panel shape:", daily_panel.shape)
print("sku_activity shape:", sku_activity.shape)

display(daily_panel.head())
display(sku_activity.head())

daily_panel shape: (28014888, 17)
sku_activity shape: (15972, 20)


,ItemCode,Date,y_net,y_gross,y_return,sales,cost,profit,transaction_count,y_net_clip,dayofweek,is_saturday,is_sunday,month,day,weekofyear,year
0,SKU-00001,2020-11-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,11,17,47,2020
1,SKU-00001,2020-11-18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0,0,11,18,47,2020
2,SKU-00001,2020-11-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,0,0,11,19,47,2020
3,SKU-00001,2020-11-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0,0,11,20,47,2020
4,SKU-00001,2020-11-21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,0,11,21,47,2020


,ItemCode,active_days,active_net_days,total_y_net,total_y_gross,total_return,total_sales,total_cost,total_profit,total_transactions,first_sale_date,last_sale_date,first_transaction_date,last_transaction_date,has_ever_sold,days_since_last_sale,days_since_last_transaction,positive_profit,profit_rank,return_rate_qty
0,SKU-00001,15,15,30.0,30.0,0.0,3.608433e+07,0.0,3.608433e+07,30.0,2025-05-26,2025-08-28,2025-05-26,2025-08-28,1,8,8,3.608433e+07,781,0.0
1,SKU-00002,895,895,5894.0,5894.0,0.0,8.012686e+09,0.0,8.012686e+09,5896.0,2022-01-10,2025-09-05,2022-01-10,2025-09-05,1,0,0,8.012686e+09,2,0.0
2,SKU-00003,1061,1061,10935.0,10935.0,0.0,1.670068e+10,0.0,1.670068e+10,10936.0,2022-01-03,2025-09-04,2022-01-03,2025-09-04,1,1,1,1.670068e+10,1,0.0
3,SKU-00004,279,279,659.0,659.0,0.0,8.359968e+08,0.0,8.359968e+08,659.0,2023-07-19,2024-12-20,2023-07-19,2024-12-20,1,259,259,8.359968e+08,12,0.0
4,SKU-00005,327,327,1101.0,1101.0,0.0,2.243340e+09,0.0,2.243340e+09,1103.0,2022-01-03,2023-06-26,2022-01-03,2023-06-26,1,802,802,2.243340e+09,4,0.0


In [2]:
print("Date range:", daily_panel["Date"].min(), "→", daily_panel["Date"].max())
print("Number of SKUs:", daily_panel["ItemCode"].nunique())

print("Columns daily_panel:")
print(daily_panel.columns.tolist())

print("Columns sku_activity:")
print(sku_activity.columns.tolist())

Date range: 2020-11-17 00:00:00 → 2025-09-05 00:00:00
Number of SKUs: 15972
Columns daily_panel:
['ItemCode', 'Date', 'y_net', 'y_gross', 'y_return', 'sales', 'cost', 'profit', 'transaction_count', 'y_net_clip', 'dayofweek', 'is_saturday', 'is_sunday', 'month', 'day', 'weekofyear', 'year']
Columns sku_activity:
['ItemCode', 'active_days', 'active_net_days', 'total_y_net', 'total_y_gross', 'total_return', 'total_sales', 'total_cost', 'total_profit', 'total_transactions', 'first_sale_date', 'last_sale_date', 'first_transaction_date', 'last_transaction_date', 'has_ever_sold', 'days_since_last_sale', 'days_since_last_transaction', 'positive_profit', 'profit_rank', 'return_rate_qty']


## 2. Main target to calculate metric local

In [3]:
PRIMARY_TARGET = "y_net"
EPS = 1e-8

TRAIN_END = pd.Timestamp("2025-09-05")
FINAL_FORECAST_START = pd.Timestamp("2025-09-06")

## 3. Calculate profit_weight and scale function

In [4]:
def compute_sku_metric_info(
    panel: pd.DataFrame,
    train_end,
    target_col: str = "y_net",
    profit_col: str = "profit",
    eps: float = 1e-8
) -> pd.DataFrame:
    """
    Compute SKU-level profit weight and RMSSE scale up to train_end.
    
    weight_i = max(sum(profit_i), 0) / total_positive_profit
    scale_i = mean((y_t - y_{t-1})^2) over training history
    """
    train_end = pd.Timestamp(train_end)
    
    hist = panel.loc[
        panel["Date"] <= train_end,
        ["ItemCode", "Date", target_col, profit_col]
    ].copy()
    
    hist = hist.sort_values(["ItemCode", "Date"])
    
    # 1. Profit weight
    profit_info = (
        hist.groupby("ItemCode", as_index=False)[profit_col]
        .sum()
        .rename(columns={profit_col: "total_profit_metric_train"})
    )
    
    profit_info["positive_profit"] = profit_info["total_profit_metric_train"].clip(lower=0)
    
    total_positive_profit = profit_info["positive_profit"].sum()
    
    if total_positive_profit <= 0:
        raise ValueError("Total positive profit is zero. Cannot compute weights.")
    
    profit_info["weight"] = profit_info["positive_profit"] / total_positive_profit
    
    # 2. RMSSE scale
    hist["diff"] = hist.groupby("ItemCode")[target_col].diff()
    hist["sq_diff"] = hist["diff"] ** 2
    
    scale_info = (
        hist.dropna(subset=["sq_diff"])
        .groupby("ItemCode", as_index=False)["sq_diff"]
        .mean()
        .rename(columns={"sq_diff": "scale"})
    )
    
    metric_info = profit_info.merge(scale_info, on="ItemCode", how="left")
    
    metric_info["scale"] = metric_info["scale"].fillna(0)
    metric_info["scale_safe"] = metric_info["scale"].clip(lower=eps)
    metric_info["zero_scale_flag"] = (metric_info["scale"] < eps).astype(int)
    
    metric_info["profit_rank"] = (
        metric_info["positive_profit"]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    
    metric_info = metric_info.sort_values("profit_rank").reset_index(drop=True)
    metric_info["cum_weight"] = metric_info["weight"].cumsum()
    
    return metric_info

## 4. Calculate metric info for full train

In [5]:
sku_metric_info = compute_sku_metric_info(
    daily_panel,
    train_end=TRAIN_END,
    target_col=PRIMARY_TARGET
)

print("shape:", sku_metric_info.shape)
print("weight sum:", sku_metric_info["weight"].sum())
print("positive profit sum:", sku_metric_info["positive_profit"].sum())
print("zero scale SKUs:", sku_metric_info["zero_scale_flag"].sum())

display(sku_metric_info.head(20))
display(sku_metric_info.describe())

shape: (15972, 9)
weight sum: 0.9999999999999999
positive profit sum: 172197100139.09998
zero scale SKUs: 149


,ItemCode,total_profit_metric_train,positive_profit,weight,scale,scale_safe,zero_scale_flag,profit_rank,cum_weight
0,SKU-00003,1.670068e+10,1.670068e+10,0.096986,61.625784,61.625784,0,1,0.096986
1,SKU-00002,8.012686e+09,8.012686e+09,0.046532,24.374216,24.374216,0,2,0.143518
2,SKU-09458,2.641121e+09,2.641121e+09,0.015338,137530.913862,137530.913862,0,3,0.158856
3,SKU-00005,2.243340e+09,2.243340e+09,0.013028,4.423274,4.423274,0,4,0.171883
4,SKU-08589,1.285705e+09,1.285705e+09,0.007466,38186.784940,38186.784940,0,5,0.179350
5,SKU-12534,1.215804e+09,1.215804e+09,0.007061,6092.189390,6092.189390,0,6,0.186410
6,SKU-09760,1.195313e+09,1.195313e+09,0.006942,379715.817456,379715.817456,0,7,0.193352
7,SKU-12537,1.106800e+09,1.106800e+09,0.006428,3586.547633,3586.547633,0,8,0.199780
8,SKU-00324,9.783747e+08,9.783747e+08,0.005682,265.025670,265.025670,0,9,0.205461
9,SKU-14323,9.371040e+08,9.371040e+08,0.005442,1804.553337,1804.553337,0,10,0.210903


,total_profit_metric_train,positive_profit,weight,scale,scale_safe,zero_scale_flag,profit_rank,cum_weight
count,1.597200e+04,1.597200e+04,15972.000000,15972.000000,1.597200e+04,15972.000000,15972.000000,15972.000000
mean,1.058337e+07,1.078119e+07,0.000063,53.671591,5.367159e+01,0.009329,7882.603681,0.940962
std,1.543916e+08,1.543365e+08,0.000896,3257.277406,3.257277e+03,0.096137,4454.656778,0.108636
min,-3.028037e+08,0.000000e+00,0.000000,0.000000,1.000000e-08,0.000000,1.000000,0.096986
25%,1.792658e+05,1.792658e+05,0.000001,0.004564,4.563605e-03,0.000000,3993.750000,0.936445
50%,9.540000e+05,9.540000e+05,0.000006,0.018254,1.825442e-02,0.000000,7986.000000,0.987628
75%,4.463201e+06,4.463201e+06,0.000026,0.132345,1.323446e-01,0.000000,11979.250000,0.998846
max,1.670068e+10,1.670068e+10,0.096986,379715.817456,3.797158e+05,1.000000,14157.000000,1.000000


In [6]:
for k in [10, 50, 100, 160, 500, 1000, 2000]:
    top_k_weight = sku_metric_info.head(k)["weight"].sum()
    print(f"Top {k} cumulative weight: {top_k_weight:.4f}")

Top 10 cumulative weight: 0.2109
Top 50 cumulative weight: 0.3225
Top 100 cumulative weight: 0.3982
Top 160 cumulative weight: 0.4595
Top 500 cumulative weight: 0.6338
Top 1000 cumulative weight: 0.7473
Top 2000 cumulative weight: 0.8518


## 5. Save metric info

In [7]:
metric_info_path = DATA_PROCESSED / f"sku_metric_info_{PRIMARY_TARGET}.parquet"
sku_metric_info.to_parquet(metric_info_path, index=False)

print("Saved:", metric_info_path)

Saved: ../data/processed/sku_metric_info_y_net.parquet


## 6. Create actual matrix for validation function

In [8]:
def make_actual_matrix(
    panel: pd.DataFrame,
    start_date,
    horizon: int = 56,
    target_col: str = "y_net"
) -> pd.DataFrame:
    """
    Create actual matrix:
    index = ItemCode
    columns = validation dates
    values = actual target
    """
    start_date = pd.Timestamp(start_date)
    dates = pd.date_range(start_date, periods=horizon, freq="D")
    
    subset = panel.loc[
        panel["Date"].isin(dates),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    actual_wide = (
        subset.pivot(index="ItemCode", columns="Date", values=target_col)
        .sort_index()
        .reindex(columns=dates)
    )
    
    if actual_wide.isna().any().any():
        raise ValueError("Actual matrix contains NaN. Check date range or panel completeness.")
    
    return actual_wide

## 7. WRMSSE function

In [9]:
def wrmsse_score(
    actual_wide: pd.DataFrame,
    pred_wide: pd.DataFrame,
    metric_info: pd.DataFrame,
    clip_pred: bool = True
):
    """
    Compute WRMSSE.
    
    actual_wide:
        index = ItemCode
        columns = forecast dates
    pred_wide:
        same shape as actual_wide
    metric_info:
        must contain ItemCode, weight, scale_safe
    """
    actual_wide = actual_wide.sort_index()
    
    pred_wide = (
        pred_wide
        .reindex(index=actual_wide.index, columns=actual_wide.columns)
        .fillna(0)
        .sort_index()
    )
    
    actual_values = actual_wide.to_numpy(dtype=float)
    pred_values = pred_wide.to_numpy(dtype=float)
    
    if clip_pred:
        pred_values = np.clip(pred_values, 0, None)
    
    mse = np.mean((actual_values - pred_values) ** 2, axis=1)
    
    detail = pd.DataFrame({
        "ItemCode": actual_wide.index,
        "mse": mse
    })
    
    detail = detail.merge(
        metric_info[[
            "ItemCode",
            "weight",
            "scale",
            "scale_safe",
            "total_profit_metric_train",
            "positive_profit",
            "profit_rank"
        ]],
        on="ItemCode",
        how="left"
    )
    
    if detail["weight"].isna().any():
        raise ValueError("Some SKUs in actual_wide are missing from metric_info.")
    
    detail["rmsse"] = np.sqrt(detail["mse"] / detail["scale_safe"])
    detail["weighted_rmsse"] = detail["weight"] * detail["rmsse"]
    
    score = detail["weighted_rmsse"].sum()
    
    return score, detail

## 8. Basic baseline for metric testing

### Baseline 0: forecast all 0

In [10]:
def make_zero_prediction(actual_wide: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        0.0,
        index=actual_wide.index,
        columns=actual_wide.columns
    )

### Baseline 1: mean last 28 days

In [11]:
def make_recent_mean_prediction(
    panel: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    history_target_col: str = "y_net_clip",
    window: int = 28
) -> pd.DataFrame:
    """
    Predict each future day by SKU's mean demand over the last `window` days.
    Use y_net_clip by default because prediction cannot be negative.
    """
    train_end = pd.Timestamp(train_end)
    forecast_start = pd.Timestamp(forecast_start)
    
    hist_start = train_end - pd.Timedelta(days=window - 1)
    forecast_dates = pd.date_range(forecast_start, periods=horizon, freq="D")
    
    hist = panel.loc[
        (panel["Date"] >= hist_start) &
        (panel["Date"] <= train_end),
        ["ItemCode", history_target_col]
    ].copy()
    
    avg_by_sku = hist.groupby("ItemCode")[history_target_col].mean()
    
    itemcodes = sorted(panel["ItemCode"].unique())
    
    values = avg_by_sku.reindex(itemcodes).fillna(0).to_numpy(dtype=float)
    
    pred = pd.DataFrame(
        np.repeat(values[:, None], horizon, axis=1),
        index=itemcodes,
        columns=forecast_dates
    )
    
    return pred

Sunday correction

In [12]:
def apply_sunday_factor(pred_wide: pd.DataFrame, sunday_factor: float = 0.0) -> pd.DataFrame:
    """
    Reduce predictions on Sundays.
    sunday_factor = 0.0 means force Sunday forecasts to zero.
    """
    pred = pred_wide.copy()
    
    sunday_cols = [c for c in pred.columns if pd.Timestamp(c).dayofweek == 6]
    
    if sunday_cols:
        pred.loc[:, sunday_cols] = pred.loc[:, sunday_cols] * sunday_factor
    
    return pred

## 9. Setup 3 validation folds

In [13]:
folds = [
    {
        "fold": "recent_2025",
        "train_end": "2025-07-11",
        "valid_start": "2025-07-12",
        "horizon": 56
    },
    {
        "fold": "seasonal_2024",
        "train_end": "2024-09-05",
        "valid_start": "2024-09-06",
        "horizon": 56
    },
    {
        "fold": "seasonal_2023",
        "train_end": "2023-09-05",
        "valid_start": "2023-09-06",
        "horizon": 56
    }
]

for f in folds:
    valid_end = pd.Timestamp(f["valid_start"]) + pd.Timedelta(days=f["horizon"] - 1)
    print(f["fold"], "| train_end:", f["train_end"], "| valid:", f["valid_start"], "→", valid_end.date())

recent_2025 | train_end: 2025-07-11 | valid: 2025-07-12 → 2025-09-05
seasonal_2024 | train_end: 2024-09-05 | valid: 2024-09-06 → 2024-10-31
seasonal_2023 | train_end: 2023-09-05 | valid: 2023-09-06 → 2023-10-31


## 10. Test WRMSSE on baselines

In [14]:
cv_records = []
detail_outputs = {}

for fold_cfg in folds:
    fold_name = fold_cfg["fold"]
    train_end = fold_cfg["train_end"]
    valid_start = fold_cfg["valid_start"]
    horizon = fold_cfg["horizon"]
    
    print("=" * 80)
    print("Fold:", fold_name)
    
    metric_info_fold = compute_sku_metric_info(
        daily_panel,
        train_end=train_end,
        target_col=PRIMARY_TARGET
    )
    
    actual_wide = make_actual_matrix(
        daily_panel,
        start_date=valid_start,
        horizon=horizon,
        target_col=PRIMARY_TARGET
    )
    
    # Baseline 0: zero
    pred_zero = make_zero_prediction(actual_wide)
    score_zero, detail_zero = wrmsse_score(actual_wide, pred_zero, metric_info_fold)
    
    cv_records.append({
        "fold": fold_name,
        "target_col": PRIMARY_TARGET,
        "model": "zero",
        "wrmsse": score_zero
    })
    
    detail_outputs[(fold_name, "zero")] = detail_zero
    
    print("zero WRMSSE:", score_zero)
    
    # Baseline 1: recent mean 28
    pred_recent28 = make_recent_mean_prediction(
        daily_panel,
        train_end=train_end,
        forecast_start=valid_start,
        horizon=horizon,
        history_target_col="y_net_clip",
        window=28
    )
    
    score_recent28, detail_recent28 = wrmsse_score(actual_wide, pred_recent28, metric_info_fold)
    
    cv_records.append({
        "fold": fold_name,
        "target_col": PRIMARY_TARGET,
        "model": "recent_mean_28",
        "wrmsse": score_recent28
    })
    
    detail_outputs[(fold_name, "recent_mean_28")] = detail_recent28
    
    print("recent_mean_28 WRMSSE:", score_recent28)
    
    # Baseline 2: recent mean 28 + Sunday zero
    pred_recent28_sun0 = apply_sunday_factor(pred_recent28, sunday_factor=0.0)
    
    score_recent28_sun0, detail_recent28_sun0 = wrmsse_score(
        actual_wide,
        pred_recent28_sun0,
        metric_info_fold
    )
    
    cv_records.append({
        "fold": fold_name,
        "target_col": PRIMARY_TARGET,
        "model": "recent_mean_28_sunday_zero",
        "wrmsse": score_recent28_sun0
    })
    
    detail_outputs[(fold_name, "recent_mean_28_sunday_zero")] = detail_recent28_sun0
    
    print("recent_mean_28_sunday_zero WRMSSE:", score_recent28_sun0)

cv_results = pd.DataFrame(cv_records)
display(cv_results)

Fold: recent_2025
zero WRMSSE: 0.665584499539824
recent_mean_28 WRMSSE: 0.5906896110221014
recent_mean_28_sunday_zero WRMSSE: 0.5671544459095139
Fold: seasonal_2024
zero WRMSSE: 0.8971165611918634
recent_mean_28 WRMSSE: 0.7743416729452195
recent_mean_28_sunday_zero WRMSSE: 0.7504267513598499
Fold: seasonal_2023
zero WRMSSE: 0.9275827721561002
recent_mean_28 WRMSSE: 0.8546102657031436
recent_mean_28_sunday_zero WRMSSE: 0.8260205318173562


,fold,target_col,model,wrmsse
0,recent_2025,y_net,zero,0.665584
1,recent_2025,y_net,recent_mean_28,0.590690
2,recent_2025,y_net,recent_mean_28_sunday_zero,0.567154
3,seasonal_2024,y_net,zero,0.897117
4,seasonal_2024,y_net,recent_mean_28,0.774342
5,seasonal_2024,y_net,recent_mean_28_sunday_zero,0.750427
6,seasonal_2023,y_net,zero,0.927583
7,seasonal_2023,y_net,recent_mean_28,0.854610
8,seasonal_2023,y_net,recent_mean_28_sunday_zero,0.826021


## 11. See results with pivot

In [15]:
cv_pivot = cv_results.pivot_table(
    index="model",
    columns="fold",
    values="wrmsse"
)

cv_pivot["mean_wrmsse"] = cv_pivot.mean(axis=1)
cv_pivot = cv_pivot.sort_values("mean_wrmsse")

display(cv_pivot)

fold,recent_2025,seasonal_2023,seasonal_2024,mean_wrmsse
model,,,,
recent_mean_28_sunday_zero,0.567154,0.826021,0.750427,0.714534
recent_mean_28,0.590690,0.854610,0.774342,0.739881
zero,0.665584,0.927583,0.897117,0.830095


In [16]:
cv_results_path = CV_DIR / "step3_dummy_baselines.csv"
cv_results.to_csv(cv_results_path, index=False)

print("Saved:", cv_results_path)

Saved: ../outputs/cv_results/step3_dummy_baselines.csv


## 12. Diagnostic: which SKU contribute the most errors?

In [17]:
# Chọn fold recent và baseline recent_mean_28_sunday_zero 
diagnostic_detail = detail_outputs[("recent_2025", "recent_mean_28_sunday_zero")].copy()

display(
    diagnostic_detail
    .sort_values("weighted_rmsse", ascending=False)
    .head(30)
)

,ItemCode,mse,weight,scale,scale_safe,total_profit_metric_train,positive_profit,profit_rank,rmsse,weighted_rmsse
2,SKU-00003,91.142857,0.095070,56.281674,56.281674,1.587671e+10,1.587671e+10,1,1.272559,0.120982
1,SKU-00002,31.361516,0.045248,22.822039,22.822039,7.556427e+09,7.556427e+09,2,1.172253,0.053042
13993,SKU-14323,5324.251822,0.005414,1509.051267,1509.051267,9.041417e+08,9.041417e+08,10,1.878353,0.010169
15241,SKU-15599,17919.429665,0.002707,3148.621685,3148.621685,4.520825e+08,4.520825e+08,26,2.385623,0.006458
13990,SKU-14320,2196.131013,0.005120,1623.653506,1623.653506,8.549961e+08,8.549961e+08,11,1.163007,0.005954
10248,SKU-10532,141.642857,0.002977,70.189747,70.189747,4.972008e+08,4.972008e+08,22,1.420563,0.004229
11100,SKU-11398,1712.428571,0.000676,47.646435,47.646435,1.128099e+08,1.128099e+08,222,5.995025,0.004050
8616,SKU-08863,1627.736516,0.003742,1627.361226,1627.361226,6.249812e+08,6.249812e+08,17,1.000115,0.003743
6593,SKU-06772,183.818149,0.000679,6.265174,6.265174,1.133406e+08,1.133406e+08,220,5.416611,0.003676
9839,SKU-10117,1844.153972,0.001124,298.143783,298.143783,1.876291e+08,1.876291e+08,114,2.487057,0.002794


In [18]:
def assign_rank_bucket(rank):
    if rank <= 100:
        return "top_100"
    elif rank <= 500:
        return "top_500"
    elif rank <= 1000:
        return "top_1000"
    elif rank <= 2000:
        return "top_2000"
    else:
        return "long_tail"

diagnostic_detail["rank_bucket"] = diagnostic_detail["profit_rank"].apply(assign_rank_bucket)

bucket_summary = (
    diagnostic_detail.groupby("rank_bucket", as_index=False)
    .agg(
        sku_count=("ItemCode", "count"),
        weight_sum=("weight", "sum"),
        wrmsse_contribution=("weighted_rmsse", "sum"),
        avg_rmsse=("rmsse", "mean")
    )
    .sort_values("wrmsse_contribution", ascending=False)
)

display(bucket_summary)

,rank_bucket,sku_count,weight_sum,wrmsse_contribution,avg_rmsse
1,top_100,100,0.397870,0.272168,0.467652
4,top_500,400,0.235905,0.131046,0.559117
2,top_1000,500,0.114096,0.059074,0.503293
0,long_tail,13972,0.146865,0.057545,89.389702
3,top_2000,1000,0.105264,0.047321,0.442978


## 13. Check first 28 days and last 28 days

In [19]:
def wrmsse_score_by_horizon_split(
    actual_wide: pd.DataFrame,
    pred_wide: pd.DataFrame,
    metric_info: pd.DataFrame
):
    actual_first = actual_wide.iloc[:, :28]
    pred_first = pred_wide.iloc[:, :28]
    
    actual_second = actual_wide.iloc[:, 28:56]
    pred_second = pred_wide.iloc[:, 28:56]
    
    score_first, detail_first = wrmsse_score(actual_first, pred_first, metric_info)
    score_second, detail_second = wrmsse_score(actual_second, pred_second, metric_info)
    
    return score_first, score_second, detail_first, detail_second

In [20]:
fold_cfg = folds[0]

metric_info_fold = compute_sku_metric_info(
    daily_panel,
    train_end=fold_cfg["train_end"],
    target_col=PRIMARY_TARGET
)

actual_wide = make_actual_matrix(
    daily_panel,
    start_date=fold_cfg["valid_start"],
    horizon=56,
    target_col=PRIMARY_TARGET
)

pred_recent28 = make_recent_mean_prediction(
    daily_panel,
    train_end=fold_cfg["train_end"],
    forecast_start=fold_cfg["valid_start"],
    horizon=56,
    history_target_col="y_net_clip",
    window=28
)

pred_recent28_sun0 = apply_sunday_factor(pred_recent28, sunday_factor=0.0)

score_first28, score_second28, _, _ = wrmsse_score_by_horizon_split(
    actual_wide,
    pred_recent28_sun0,
    metric_info_fold
)

print("Recent fold | first 28 days WRMSSE:", score_first28)
print("Recent fold | second 28 days WRMSSE:", score_second28)

Recent fold | first 28 days WRMSSE: 0.500563389037773
Recent fold | second 28 days WRMSSE: 0.5422567858968468
